<h1 style="text-align:center;">Kuhn Poker : game engine</h1>


Kuhn Poker is a two-player card game played with a deck of three cards (Jack, Queen, and King). Each player receives one private card, places an ante, and then has the opportunity to bet, check, call, or fold before the winner is determined by the highest card at showdown.

## Sumary :

* [**1. Objects of the game**](#1)
    * [1.1. Players and cards](#1_1)
    * [1.2. Chance space](#1_2)
    * [1.3. Histories](#1_3)
    * [1.4. Action correspondence](#1_4)
    * [1.5. Transition map](#1_5)
    * [1.6. Player function](#1_6)
    * [1.7. Utility function](#1_7)
    * [1.8. Information sets](#1_8)
    * [1.9. State object](#1_9)
* [**2. RUN**](#2)

In [15]:
# Modules
from itertools import permutations
from dataclasses import dataclass
from typing import Optional
import numpy as np  

<a id='1'></a>
# 1. Objects of the game

<a id='1_1'></a>
### 1.1. Players and cards

- Player set: $\qquad N=\{0,1\}$

- Card set : $ \qquad \mathcal{C}=\{J,Q,K\}$

- Rank map: $ \qquad r:\mathcal{C}\to\{0,1,2\}, \qquad r(J)=0,\quad r(Q)=1,\quad r(K)=2$

In [16]:
PLAYERS = [0, 1]
CARDS = ["J", "Q", "K"]
CARD_RANK = {"J": 0, "Q": 1, "K": 2}

<a id='1_2'></a>
### 1.2. Chance space

- Chance outcome is an ordered deal: $\qquad \omega=(c_0,c_1)$

- Chance space is: $\qquad \Omega=\{(c_0,c_1)\in\mathcal{C}^2:c_0\neq c_1\}$

Since $\quad |\mathcal{C}|=3$: $|\Omega|=3\times 2=6$ :

- Chance distribution is uniform: $\qquad \rho(\omega)=\frac{1}{6},\qquad \omega\in\Omega$

In [17]:
OMEGA = list(permutations(CARDS, 2))
CHANCE_PROB = {omega: 1 / len(OMEGA) for omega in OMEGA}

<a id='1_3'></a>
### 1.3. Histories

- Action alphabet: $\qquad \mathcal{A}=\{c,b,f\}$

where $c$ denotes check/call, $b$ denotes bet, and $f$ denotes fold.

- Non-terminal histories: $\qquad H\setminus Z=\{\emptyset,c,b,cb\}$

- Terminal histories: $\qquad Z=\{cc,bc,bf,cbc,cbf\}$

- Full public-history set: $\qquad h=(H\setminus Z)\cup Z$

In [18]:
CHECK_CALL = "c"
BET = "b"
FOLD = "f"

ACTIONS = [CHECK_CALL, BET, FOLD]

NON_TERMINAL_HISTORIES = {"", "c", "b", "cb"}
TERMINAL_HISTORIES = {"cc", "bc", "bf", "cbc", "cbf"}
HISTORIES = NON_TERMINAL_HISTORIES | TERMINAL_HISTORIES


def is_terminal(history: str) -> bool:
    '''Indicator of z in Z.'''
    return history in TERMINAL_HISTORIES

<a id='1_4'></a>
### 1.4. Action correspondence

The legal-action correspondence is:

$$
A(h)=
\begin{cases}
\{c,b\}, & h\in\{\emptyset,c\},\\
\{c,f\}, & h\in\{b,cb\},\\
\emptyset, & h\in Z.
\end{cases}
$$


In [19]:
def legal_actions(history: str) -> list[str]:
    '''Return A(h).'''
    if history in ("", "c"):
        return ["c", "b"]
    if history in ("b", "cb"):
        return ["c", "f"]
    if is_terminal(history):
        return []

<a id='1_5'></a>
### 1.5. Transition map

For non-terminal $h$ and legal action $a\in A(h)$:

$$
\tau(h,a)=ha.
$$

The deterministic edges are:

<div align="center">

```text

          ∅
          /   \
         c     b
       /  \   / \
     cc   cb bc bf
         / \
      cbc cbf

In [20]:
def next_history(history: str, action: str) -> str:
    '''Return tau(h,a)=ha after checking a in A(h).'''
    return history + action

<a id='1_6'></a>
### 1.6. Player function

*Convention : Player 0 always begin*

The player-to-act function is:

$$
P:H\setminus Z\to N
$$

with:

$$
P(h)=
\begin{cases}
0, & h\in\{\emptyset,cb\},\\
1, & h\in\{c,b\}.
\end{cases}
$$

In [21]:
def current_player(history: str) -> Optional[int]:
    '''Return P(h), or None for h in Z.'''
    if is_terminal(history):
        return None
    if history in ("", "cb"):
        return 0
    if history in ("c", "b"):
        return 1

<a id='1_7'></a>
### 1.7. Utility function

Let $\omega=(c_0,c_1)\in\Omega$.

Define:

$$
s_0(\omega)=
\begin{cases}
1, & r(c_0)>r(c_1),\\
-1, & r(c_0)<r(c_1).
\end{cases}
$$

Player $0$'s terminal utility is:

$$
u_0(\omega,z)=
\begin{cases}
s_0(\omega), & z=cc,\\
2s_0(\omega), & z\in\{bc,cbc\},\\
1, & z=bf,\\
-1, & z=cbf.
\end{cases}
$$

The game is zero-sum:

$$
u_1(\omega,z)=-u_0(\omega,z).
$$

In [22]:
def showdown_sign_player0(cards: tuple[str, str]) -> int:
    '''Return s_0(omega).'''
    c0, c1 = cards
    return 1 if CARD_RANK[c0] > CARD_RANK[c1] else -1


def payoff_player0(cards: tuple[str, str], history: str) -> int:
    '''Return u_0(omega,z).'''
    if not is_terminal(history):
        raise ValueError(f"Payoff is defined only for z in Z, got {history!r}")
    if history == "cc":
        return showdown_sign_player0(cards)
    if history in ("bc", "cbc"):
        return 2 * showdown_sign_player0(cards)
    if history == "bf":
        return 1
    if history == "cbf":
        return -1


def payoff(cards: tuple[str, str], history: str, player: int = 0) -> int:
    '''Return u_i(omega,z).'''
    u0 = payoff_player0(cards, history)
    return u0 if player == 0 else -u0

<a id='1_8'></a>
### 1.8. Information sets : Input construction

For player $i$, two decision states $(\omega,h)$ and $(\omega',h')$ are indistinguishable if:

$$
(\omega,h)\sim_i(\omega',h')
\Longleftrightarrow
h=h'
\quad\text{and}\quad
\omega_i=\omega'_i.
$$

The information set containing $(\omega,h)$ is:

$$
I_i(\omega_i,h)
=
\{(\omega',h):\omega'_i=\omega_i,\ P(h)=i\}.
$$

We encode it by:

$$
\texttt{P\{i\}|\{card\}|\{history\}}.
$$

In [23]:
def infoset_key(cards: tuple[str, str], history: str) -> Optional[str]:
    '''Return the key representing I_i(card_i,h).'''
    player = current_player(history)
    if player is None:
        return None
    private_card = cards[player]
    return f"P{player}|{private_card}|{history}"

<a id='1_9'></a>
### 1.9. State object

A state is:

$$
x=(\omega,h)\in\Omega\times H.
$$

It bundles the previously defined maps:

$$
P(h),\quad A(h),\quad \tau(h,a),\quad u_i(\omega,h),\quad I_i(\omega_i,h).
$$

In [24]:
@dataclass(frozen=True)
class State:
    cards: tuple[str, str]
    history: str = ""

    @property
    def player(self) -> Optional[int]:
        return current_player(self.history)

    @property
    def terminal(self) -> bool:
        return is_terminal(self.history)

    def actions(self) -> list[str]:
        return legal_actions(self.history)

    def child(self, action: str) -> "State":
        return State(cards=self.cards, history=next_history(self.history, action))

    def infoset_key(self) -> Optional[str]:
        return infoset_key(self.cards, self.history)

    def utility(self, player: int = 0) -> int:
        return payoff(self.cards, self.history, player=player)


s = State(cards=("K", "J"))

<a id='2'></a>
# 2. RUN

In [29]:
def run_game(
    strategy_0: dict[str, dict[str, float]],
    strategy_1: dict[str, dict[str, float]],
    seed: Optional[int] = None,
    verbose: bool = True,
) -> int:
    """
    Simulate one game of Kuhn Poker between two strategies.

    Parameters
    ----------
    strategy_0 : dict
        Strategy used by player 0.
        Maps each information-set key to action probabilities.

    strategy_1 : dict
        Strategy used by player 1.
        Maps each information-set key to action probabilities.

    seed : Optional[int]
        Random seed used to make the simulation reproducible.

    verbose : bool
        If True, display the different steps of the game.

    Returns
    -------
    int
        Final utility of player 0.
    """
    rng = np.random.default_rng(seed)

    # Deal the cards randomly
    cards = OMEGA[rng.integers(len(OMEGA))]

    # Create the initial game state
    state = State(cards=cards, history="")

    strategies = {
        0: strategy_0,
        1: strategy_1,
    }

    if verbose:
        print("=== New game ===")
        print(f"Cards: P0={cards[0]}, P1={cards[1]}")

    # Play until a terminal state is reached
    while not state.terminal:
        player = state.player
        actions = state.actions()
        key = state.infoset_key()

        # Select the strategy of the current player
        player_strategy = strategies[player]

        # Retrieve the action probabilities at this information set
        action_probabilities = player_strategy[key]

        probabilities = np.array(
            [action_probabilities[action] for action in actions],
            dtype=float,
        )

        # Check that the probabilities define a valid distribution
        if np.any(probabilities < 0):
            raise ValueError(
                f"Negative probability at information set {key}: "
                f"{action_probabilities}"
            )

        if not np.isclose(probabilities.sum(), 1.0):
            raise ValueError(
                f"Probabilities at information set {key} must sum to 1. "
                f"Received {action_probabilities}."
            )

        # Sample an action according to the current strategy
        action = rng.choice(actions, p=probabilities)

        if verbose:
            print(f"\nHistory: {state.history!r}")
            print(f"Current player: P{player}")
            print(f"Information set: {key}")
            print(f"Legal actions: {actions}")
            print(f"Action probabilities: {action_probabilities}")
            print(f"Selected action: {action}")

        # Move to the next state
        state = state.child(action)

    utility_0 = state.utility(player=0)
    utility_1 = state.utility(player=1)

    if verbose:
        print("\n=== End of game ===")
        print(f"Terminal history: {state.history!r}")
        print(f"Player 0 utility: {utility_0}")
        print(f"Player 1 utility: {utility_1}")

    return utility_0

In [30]:
random_strategy_0 = {
    "P0|J|": {"c": 0.5, "b": 0.5},
    "P0|Q|": {"c": 0.5, "b": 0.5},
    "P0|K|": {"c": 0.5, "b": 0.5},

    "P0|J|cb": {"c": 0.5, "f": 0.5},
    "P0|Q|cb": {"c": 0.5, "f": 0.5},
    "P0|K|cb": {"c": 0.5, "f": 0.5},
}

random_strategy_1 = {
    "P1|J|c": {"c": 0.5, "b": 0.5},
    "P1|Q|c": {"c": 0.5, "b": 0.5},
    "P1|K|c": {"c": 0.5, "b": 0.5},

    "P1|J|b": {"c": 0.5, "f": 0.5},
    "P1|Q|b": {"c": 0.5, "f": 0.5},
    "P1|K|b": {"c": 0.5, "f": 0.5},
}

utility = run_game(
    strategy_0=random_strategy_0,
    strategy_1=random_strategy_1,
    seed=42,
    verbose=True,
)

=== New game ===
Cards: P0=J, P1=Q

History: ''
Current player: P0
Information set: P0|J|
Legal actions: ['c', 'b']
Action probabilities: {'c': 0.5, 'b': 0.5}
Selected action: c

History: 'c'
Current player: P1
Information set: P1|Q|c
Legal actions: ['c', 'b']
Action probabilities: {'c': 0.5, 'b': 0.5}
Selected action: b

History: 'cb'
Current player: P0
Information set: P0|J|cb
Legal actions: ['c', 'f']
Action probabilities: {'c': 0.5, 'f': 0.5}
Selected action: f

=== End of game ===
Terminal history: 'cbf'
Player 0 utility: -1
Player 1 utility: 1
